In [116]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [117]:
import os
os.chdir('C:\Kaggle_Competition\Playground\S6E4-Irrigation-Need')
import warnings
import gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math
import seaborn as sns
import config
from src.utils import (
    read_csv_file,
    reduce_mem_usage,
    save_csv_file
)
from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import LogisticRegression
from src.fe import *
import optuna
from tqdm.notebook import tqdm
from itertools import combinations
pd.set_option('display.max_columns', None)

In [118]:
train_raw = read_csv_file(r'data\raw\train.csv')
test_raw = read_csv_file(r'data\raw\test.csv')

# Train and test ids
train_ids = train_raw['id']
test_ids = test_raw['id']

n_classes = 3

Reading file from: data\raw\train.csv
Reading file from: data\raw\test.csv


In [119]:
# X and y split
X = train_raw.drop(['id', 'irrigation_need'], axis=1)
y = train_raw['irrigation_need'].map(config.TARGET_MAP)
classes = np.unique(y)

# Test data
test_data = test_raw.drop('id', axis=1)

In [120]:
# Reduce memory size of data
X = reduce_mem_usage(X)
test_data = reduce_mem_usage(test_data)

Memory before: 354.06 MB
Memory after: 327.63 MB
Reduced by: 26.44 MB (7.5%)
Memory before: 151.74 MB
Memory after: 140.41 MB
Reduced by: 11.33 MB (7.5%)


In [121]:
# OOF
oof_proba = read_csv_file(r'artifacts\oof\Logistic_seed42_v21_oof_proba.csv')
oof_proba = oof_proba.iloc[:, 1:]

# TEST_PROBA
test_proba = read_csv_file(r'artifacts\test_proba\Logistic_seed42_v21_test_proba.csv')
test_proba = test_proba.iloc[:, 1:]

Reading file from: artifacts\oof\Logistic_seed42_v21_oof_proba.csv
Reading file from: artifacts\test_proba\Logistic_seed42_v21_test_proba.csv


In [122]:
n_classes = 3

def predict_with_thresholds(probs, thresholds):
    scaled = probs / (np.array(thresholds) + 1e-8)
    return np.argmax(scaled, axis=1)

In [123]:
def objective(trial):
    thresholds = [
        trial.suggest_float(f"thresh_{i}", 0.75, 0.95)
        for i in range(n_classes)
    ]
    preds = predict_with_thresholds(oof_proba, thresholds)
    return balanced_accuracy_score(y, preds)

In [124]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
)
study.optimize(objective, n_trials=5000, show_progress_bar=True)

[I 2026-04-22 02:27:31,365] A new study created in memory with name: no-name-89971278-a1fe-46b7-965f-3ef31864e327


  0%|          | 0/5000 [00:00<?, ?it/s]

[I 2026-04-22 02:27:31,414] Trial 0 finished with value: 0.9764702047887993 and parameters: {'thresh_0': 0.8249080237694725, 'thresh_1': 0.9401428612819832, 'thresh_2': 0.896398788362281}. Best is trial 0 with value: 0.9764702047887993.
[I 2026-04-22 02:27:31,453] Trial 1 finished with value: 0.9764679327488054 and parameters: {'thresh_0': 0.8697316968394073, 'thresh_1': 0.7812037280884873, 'thresh_2': 0.7811989040672406}. Best is trial 0 with value: 0.9764702047887993.
[I 2026-04-22 02:27:31,489] Trial 2 finished with value: 0.9764407722881194 and parameters: {'thresh_0': 0.7616167224336399, 'thresh_1': 0.923235229154987, 'thresh_2': 0.8702230023486417}. Best is trial 0 with value: 0.9764702047887993.
[I 2026-04-22 02:27:31,534] Trial 3 finished with value: 0.9764382688582464 and parameters: {'thresh_0': 0.891614515559209, 'thresh_1': 0.7541168988591604, 'thresh_2': 0.9439819704323988}. Best is trial 0 with value: 0.9764702047887993.
[I 2026-04-22 02:27:31,573] Trial 4 finished with v

In [132]:
best_thresholds = [study.best_params[f"thresh_{i}"] for i in range(n_classes)]

baseline  = balanced_accuracy_score(y, np.argmax(oof_proba, axis=1))
optimised = balanced_accuracy_score(y, predict_with_thresholds(oof_proba, best_thresholds))

abs_gain = optimised - baseline
pct_gain = (abs_gain / baseline) * 100 if baseline != 0 else 0

print(f"Baseline  BACC: {baseline}")
print(f"Optimised BACC: {optimised}")
print(f"Gain: +{abs_gain:.5f} ({pct_gain:.5f}%)")
print(f"Thresholds : class_0={best_thresholds[0]:.5f} | class_1={best_thresholds[1]:.5f} | class_2={best_thresholds[2]:.5f}")

Baseline  BACC: 0.9764723778824064
Optimised BACC: 0.9766173396198706
Gain: +0.00014 (0.01485%)
Thresholds : class_0=0.78667 | class_1=0.87123 | class_2=0.88951


In [131]:
out = predict_with_thresholds(test_proba, best_thresholds)
out = pd.DataFrame({'id': test_ids, 'Irrigation_Need': out})
out.Irrigation_Need = out.Irrigation_Need.map(config.TARGET_MAP_INV)

save_path = os.path.join(config.SUBMISSIONS_DIR, 'logistic_seed_42_v21_thresh_tuned.csv')
save_csv_file(out, save_path)

Saving file to: artifacts\submissions\logistic_seed_42_v21_thresh_tuned.csv
